In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report
import json

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
with open("/content/drive/MyDrive/I -TEAM FILES/Datasets/Facebook Comments (Features).json", 'r', encoding='utf-8') as f:
        data = json.load(f)

In [ ]:
# Parse data into a DataFrame
records = []
for entry in data:
    user_id = entry['userID']
    label = entry['label']
    features = entry['features']

    record = {'userID': user_id, 'label': label}
    if features:
        for key, value in features.items():
            record[key] = value
    records.append(record)

df = pd.DataFrame(records)

# Separate labeled and unlabeled
labeled_df = df[df['label'].notnull()]
unlabeled_df = df[df['label'].isnull()]

In [ ]:
len(labeled_df)

840

In [ ]:
labeled_df

,userID,label,averageCommentLength,averageLinkUsage,averageResponseTime,engagement,averagePairwiseCommentDisimilarity,threadDeviation,innovationRate,averageLikesPerComment,replyRatio,commentTimeVariance,uniquePostsCommentedOn,uniqueNewsOutletsCommentedOn,averageCommentsPerPost,duplicateCommentRate,averageMediaUsage,numOfComments
101,8740651,Human,33.000000,0,0.000000,0,0.000000,52360.826718,0.000000,0.000000,0.000000,0.000000e+00,1,1,1.000000,0.0,0.000000,1
518,2248813,Human,84.580645,0,303.161290,562,0.969861,35604.310147,0.633110,18.129032,0.290323,7.426395e+08,22,4,1.409091,0.0,0.032258,31
527,7548584,Human,53.000000,0,127.000000,1,1.000000,32943.760298,1.000000,0.500000,0.500000,4.000000e+02,1,1,2.000000,0.0,0.000000,2
709,2270002,Cyborg,43.461538,0,2056.615385,5,0.951077,35363.893365,0.797468,0.384615,0.230769,3.092971e+08,10,3,1.300000,0.0,0.076923,13
817,4673897,Troll,51.680000,0,1228.740000,26,0.987400,42397.900747,0.692841,0.520000,0.220000,7.970375e+08,42,6,1.190476,0.0,0.060000,50
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
319130,4528173,Human,54.000000,0,0.000000,0,0.929120,10440.173282,0.894737,0.000000,0.000000,5.163500e+03,1,1,4.000000,0.0,0.000000,4
320655,8667051,Human,93.000000,0,0.000000,0,0.000000,33010.826718,1.000000,0.000000,0.000000,0.000000e+00,1,1,1.000000,0.0,0.000000,1
321106,2285985,Cyborg,0.000000,0,0.000000,0,0.000000,42253.826718,0.000000,0.000000,0.000000,0.000000e+00,1,1,1.000000,0.0,1.000000,1
321916,8117526,Human,19.125000,0,0.000000,7,0.973014,52915.826718,0.892857,0.875000,0.000000,7.084902e+08,2,1,4.000000,0.0,0.375000,8


In [ ]:
len(unlabeled_df)

324480

In [ ]:
# Define feature columns
feature_cols = [
    "averageCommentLength",
    "averageLinkUsage",
    "averageResponseTime",
    "engagement",
    "averagePairwiseCommentDisimilarity",
    "threadDeviation",
    "innovationRate",
    "averageLikesPerComment",
    "replyRatio",
    "commentTimeVariance",
    "uniquePostsCommentedOn",
    "uniqueNewsOutletsCommentedOn",
    "averageCommentsPerPost",
    "duplicateCommentRate",
    "averageMediaUsage",
    "numOfComments"
]

###SMOTE

In [ ]:
feature_cols_SMOTE = [
    "averageCommentLength",
    "averageLinkUsage",
    "averageResponseTime",
    "engagement",
    "averagePairwiseCommentDisimilarity",
    "threadDeviation",
    "innovationRate",
    "averageLikesPerComment",
    "replyRatio",
    "commentTimeVariance",
    "uniquePostsCommentedOn",
    "uniqueNewsOutletsCommentedOn",
    "averageCommentsPerPost",
    "duplicateCommentRate",
    "averageMediaUsage",
    "numOfComments"
]

In [ ]:
X_SMOTE = []
y_SMOTE = []

# Correctly iterate over rows of the DataFrame
for _, record in labeled_df.iterrows():
    X_SMOTE.append([record.get(col) for col in feature_cols])
    y_SMOTE.append(record["label"])

In [ ]:
label_encoder = LabelEncoder()
y_SMOTE = label_encoder.fit_transform(y_SMOTE)

In [ ]:
print("Class distribution before SMOTE:", pd.Series(y_SMOTE).value_counts())

Class distribution before SMOTE: 1    452
3    206
0    156
2     26
Name: count, dtype: int64


In [ ]:
from imblearn.over_sampling import SMOTE
smote = SMOTE(random_state=42)

# Fit SMOTE to the data and generate synthetic samples
X_resampled, y_resampled = smote.fit_resample(X_SMOTE, y_SMOTE)

In [ ]:
X_train_SMOTE, X_test_SMOTE, y_train_SMOTE, y_test_SMOTE = train_test_split(X_resampled, y_resampled, test_size=0.2, random_state=42)

In [ ]:
print("Class distribution after SMOTE:", pd.Series(y_resampled).value_counts())

Class distribution after SMOTE: 1    452
0    452
3    452
2    452
Name: count, dtype: int64


In [ ]:
from sklearn.model_selection import GridSearchCV

# Define the parameter grid
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [None, 10, 20, 30],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 'log2']
}

# Initialize RandomForestClassifier
rf = RandomForestClassifier(random_state=42)

# Use GridSearchCV to find the best parameters
grid_search = GridSearchCV(estimator=rf, param_grid=param_grid, cv=5, n_jobs=-1, verbose=2)

# Fit the model
grid_search.fit(X_train_SMOTE, y_train_SMOTE)

# Get the best parameters and best score
print(f"Best Parameters: {grid_search.best_params_}")
print(f"Best Cross-Validation Accuracy: {grid_search.best_score_:.2%}")

# Use the best model for predictions
best_rf = grid_search.best_estimator_

# Evaluate on the test set
y_pred_SMOTE = best_rf.predict(X_test_SMOTE)
accuracy = accuracy_score(y_test_SMOTE, y_pred_SMOTE)
print(f"Test Set Accuracy: {accuracy:.2%}")


Fitting 5 folds for each of 216 candidates, totalling 1080 fits
Best Parameters: {'max_depth': None, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 200}
Best Cross-Validation Accuracy: 80.84%
Test Set Accuracy: 83.70%


In [ ]:
print("Classification report:\n", classification_report(y_test_SMOTE, y_pred_SMOTE, target_names=label_encoder.classes_))

Classification report:
               precision    recall  f1-score   support

      Cyborg       0.79      0.84      0.81        80
       Human       0.73      0.80      0.77        90
    Spam Bot       0.98      1.00      0.99        92
       Troll       0.85      0.72      0.78       100

    accuracy                           0.84       362
   macro avg       0.84      0.84      0.84       362
weighted avg       0.84      0.84      0.84       362



In [ ]:
#LIBRARIES

import json
import pandas as pd
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report
from sklearn.model_selection import StratifiedKFold, RandomizedSearchCV
from imblearn.over_sampling import SMOTE
from imblearn.combine import SMOTETomek
from collections import Counter

In [ ]:
# standardize features
scaler = StandardScaler()
X_train_SMOTE = scaler.fit_transform(X_train_SMOTE)
X_test_SMOTE = scaler.transform(X_test_SMOTE)

In [ ]:
# parameter grid for Randomized Search
param_dist = {
    'C': [0.1, 1, 10, 100],
    'gamma': ['scale', 'auto', 0.01, 0.1, 1],
    'kernel': ['rbf', 'poly', 'sigmoid'],
    'class_weight': [None, {0: 1, 1: 4, 2: 3, 3: 2}]
}

In [ ]:
# RandomizedSearchCV for hyperparameter tuning
svm_clf = RandomizedSearchCV(SVC(random_state=42), param_dist, n_iter=10, cv=3, scoring='f1_weighted', random_state=42)
svm_clf.fit(X_train_SMOTE, y_train_SMOTE)

RandomizedSearchCV(cv=3, estimator=SVC(random_state=42),
                   param_distributions={'C': [0.1, 1, 10, 100],
                                        'class_weight': [None,
                                                         {0: 1, 1: 4, 2: 3,
                                                          3: 2}],
                                        'gamma': ['scale', 'auto', 0.01, 0.1,
                                                  1],
                                        'kernel': ['rbf', 'poly', 'sigmoid']},
                   random_state=42, scoring='f1_weighted')

In [ ]:
# EVALUATE
y_pred = svm_clf.predict(X_test_SMOTE)
print(classification_report(y_test_SMOTE, y_pred))

              precision    recall  f1-score   support

           0       0.80      0.70      0.75        80
           1       0.68      0.76      0.72        90
           2       0.97      0.99      0.98        92
           3       0.73      0.72      0.73       100

    accuracy                           0.79       362
   macro avg       0.80      0.79      0.79       362
weighted avg       0.79      0.79      0.79       362



In [ ]:
from xgboost import XGBClassifier

In [ ]:
param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [3, 6, 10],
    'learning_rate': [0.01, 0.1, 0.2],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0]
}

# === Initialize base XGBoost model ===
xgb = XGBClassifier(
    objective='multi:softmax',
    num_class=len(label_encoder.classes_),
    eval_metric='mlogloss',
    use_label_encoder=False,
    random_state=42
)

# === GridSearchCV ===
grid_search = GridSearchCV(
    estimator=xgb,
    param_grid=param_grid,
    cv=5,
    n_jobs=-1,
    verbose=2
)

grid_search.fit(X_train_SMOTE, y_train_SMOTE)

# === Evaluate best model ===
best_xgb = grid_search.best_estimator_
y_pred = best_xgb.predict(X_test_SMOTE)

print(f"\n✅ Best Parameters: {grid_search.best_params_}")
print(f"✅ Best Cross-Validation Accuracy: {grid_search.best_score_:.2%}\n")

print("✅ Test Set Accuracy:", accuracy_score(y_test_SMOTE, y_pred))
print("✅ Classification Report:\n", classification_report(y_test_SMOTE, y_pred, target_names=label_encoder.classes_))

Fitting 5 folds for each of 72 candidates, totalling 360 fits


/usr/local/lib/python3.11/dist-packages/xgboost/core.py:158: UserWarning: [00:10:58] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)



✅ Best Parameters: {'colsample_bytree': 0.8, 'learning_rate': 0.2, 'max_depth': 10, 'n_estimators': 200, 'subsample': 0.8}
✅ Best Cross-Validation Accuracy: 81.26%

✅ Test Set Accuracy: 0.8314917127071824
✅ Classification Report:
               precision    recall  f1-score   support

      Cyborg       0.78      0.82      0.80        80
       Human       0.75      0.74      0.75        90
    Spam Bot       1.00      1.00      1.00        92
       Troll       0.79      0.76      0.78       100

    accuracy                           0.83       362
   macro avg       0.83      0.83      0.83       362
weighted avg       0.83      0.83      0.83       362



###Predicting Labels for Unlabeled Data

In [ ]:
X_unlabeled = []
for _, record in unlabeled_df.iterrows():
    X_unlabeled.append([record.get(col) for col in feature_cols])


In [ ]:
predicted_labels = best_rf.predict(X_unlabeled)

In [ ]:
predicted_labels_mapped = label_encoder.inverse_transform(predicted_labels)

In [ ]:
for i in range(len(unlabeled_df)):
    unlabeled_df.at[unlabeled_df.index[i], "label"] = predicted_labels_mapped[i]


In [ ]:
unlabeled_df

,userID,label,averageCommentLength,averageLinkUsage,averageResponseTime,engagement,averagePairwiseCommentDisimilarity,threadDeviation,innovationRate,averageLikesPerComment,replyRatio,commentTimeVariance,uniquePostsCommentedOn,uniqueNewsOutletsCommentedOn,averageCommentsPerPost,duplicateCommentRate,averageMediaUsage,numOfComments
0,3613768,Human,20.000000,0,0.000000,0,0.000000,91707.173282,1.000000,0.000000,0.000000,0.000000e+00,1,1,1.0,0.0,0.0,1
1,4865579,Troll,91.142857,0,565.571429,9,0.954742,14954.868103,0.774510,1.285714,0.857143,2.919596e+06,1,1,7.0,0.0,0.0,7
2,3463084,Human,9.000000,0,0.000000,0,0.000000,31354.173282,1.000000,0.000000,0.000000,0.000000e+00,1,1,1.0,0.0,0.0,1
3,8238042,Human,89.000000,0,0.000000,1,0.000000,27150.173282,1.000000,1.000000,0.000000,0.000000e+00,1,1,1.0,0.0,0.0,1
4,8995616,Human,80.750000,0,4454.500000,2,0.986378,5361.668321,0.943396,0.500000,0.750000,6.694003e+06,1,1,4.0,0.0,0.0,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
325315,1488451,Human,3.000000,0,0.000000,0,0.000000,56138.826718,0.000000,0.000000,0.000000,0.000000e+00,1,1,1.0,0.0,0.0,1
325316,4712060,Cyborg,0.000000,0,0.000000,0,0.000000,56139.826718,0.000000,0.000000,0.000000,0.000000e+00,1,1,1.0,0.0,1.0,1
325317,9164321,Cyborg,0.000000,0,0.000000,0,0.000000,56140.826718,0.000000,0.000000,0.000000,0.000000e+00,1,1,1.0,0.0,1.0,1
325318,2672593,Human,65.000000,0,0.000000,0,0.000000,56144.826718,1.000000,0.000000,0.000000,0.000000e+00,1,1,1.0,0.0,0.0,1


In [ ]:
labeled_df

,userID,label,averageCommentLength,averageLinkUsage,averageResponseTime,engagement,averagePairwiseCommentDisimilarity,threadDeviation,innovationRate,averageLikesPerComment,replyRatio,commentTimeVariance,uniquePostsCommentedOn,uniqueNewsOutletsCommentedOn,averageCommentsPerPost,duplicateCommentRate,averageMediaUsage,numOfComments
101,8740651,Human,33.000000,0,0.000000,0,0.000000,52360.826718,0.000000,0.000000,0.000000,0.000000e+00,1,1,1.000000,0.0,0.000000,1
518,2248813,Human,84.580645,0,303.161290,562,0.969861,35604.310147,0.633110,18.129032,0.290323,7.426395e+08,22,4,1.409091,0.0,0.032258,31
527,7548584,Human,53.000000,0,127.000000,1,1.000000,32943.760298,1.000000,0.500000,0.500000,4.000000e+02,1,1,2.000000,0.0,0.000000,2
709,2270002,Cyborg,43.461538,0,2056.615385,5,0.951077,35363.893365,0.797468,0.384615,0.230769,3.092971e+08,10,3,1.300000,0.0,0.076923,13
817,4673897,Troll,51.680000,0,1228.740000,26,0.987400,42397.900747,0.692841,0.520000,0.220000,7.970375e+08,42,6,1.190476,0.0,0.060000,50
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
319130,4528173,Human,54.000000,0,0.000000,0,0.929120,10440.173282,0.894737,0.000000,0.000000,5.163500e+03,1,1,4.000000,0.0,0.000000,4
320655,8667051,Human,93.000000,0,0.000000,0,0.000000,33010.826718,1.000000,0.000000,0.000000,0.000000e+00,1,1,1.000000,0.0,0.000000,1
321106,2285985,Cyborg,0.000000,0,0.000000,0,0.000000,42253.826718,0.000000,0.000000,0.000000,0.000000e+00,1,1,1.000000,0.0,1.000000,1
321916,8117526,Human,19.125000,0,0.000000,7,0.973014,52915.826718,0.892857,0.875000,0.000000,7.084902e+08,2,1,4.000000,0.0,0.375000,8


In [ ]:
# Combine labeled and unlabeled DataFrames
combined_df = pd.concat([labeled_df, unlabeled_df], ignore_index=True)

# List of feature columns
feature_cols = [
    "averageCommentLength",
    "averageLinkUsage",
    "averageResponseTime",
    "engagement",
    "averagePairwiseCommentDisimilarity",
    "threadDeviation",
    "innovationRate",
    "averageLikesPerComment",
    "replyRatio",
    "commentTimeVariance",
    "uniquePostsCommentedOn",
    "uniqueNewsOutletsCommentedOn",
    "averageCommentsPerPost",
    "duplicateCommentRate",
    "averageMediaUsage",
    "numOfComments"
]

# Reconstruct nested structure
nested_data = []
for _, row in combined_df.iterrows():
    entry = {
        "userID": row["userID"],
        "label": row["label"] if pd.notnull(row["label"]) else None,
        "features": {col: row[col] for col in feature_cols}
    }
    nested_data.append(entry)

# Save to JSON
output_path = "/content/drive/MyDrive/I -TEAM FILES/Datasets/Facebook_Comments (Fully Annotated).json"
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(nested_data, f, indent=4, ensure_ascii=False)


In [ ]:
print(len(nested_data))

325320
